In [ ]:
import os, torch, torch.nn as nn, torch.optim as optim
from torchvision import models
from torch.quantization import prepare_qat, convert, get_default_qat_qconfig
from data_preprocessing import get_data_loaders
from evaluation_metrics import evaluate_model, measure_inference_metrics,measure_model_size_and_flops

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

In [ ]:
# Load Data
data_dir = "tiny-imagenet-200"
train_loader, val_loader = get_data_loaders(data_dir, batch_size=64, num_workers=4, image_size=224)
print("len(train_loader):", len(train_loader))
print("len(val_loader):", len(val_loader))

In [ ]:
# Build Baseline Model
def build_baseline_model(num_classes=200):
    model = models.mobilenet_v2(pretrained=True)
    num_features = model.classifier[1].in_features
    model.classifier[1] = nn.Linear(num_features, num_classes)
    return model

baseline = build_baseline_model()
baseline = baseline.to(device)
print("Baseline MobileNetV2 model built.")

In [ ]:
# Train Baseline Model
def train_model(model, train_loader, num_epochs=10, lr=0.001):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * images.size(0)
        epoch_loss = running_loss / len(train_loader.dataset)
        print(f"Epoch {epoch+1}/{num_epochs}, Loss: {epoch_loss:.4f}")
    return model

print("Starting training baseline model...")
baseline = train_model(baseline, train_loader, num_epochs=10, lr=0.001)
print("Training complete.")

In [ ]:
def evaluate_model(model: nn.Module, data_ld) -> float:
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for x, y in data_ld:
            x, y = x.to(device), y.to(device)
            preds = model(x).argmax(1)
            correct += (preds == y).sum().item()
            total   += y.size(0)
    return 100.0 * correct / total

acc_fp32 = evaluate_model(baseline, val_loader)
print(f"Baseline Model Validation Accuracy: {acc_fp32:.2f}%")

In [ ]:
# ───────────────────────────────
# Model setup for QAT
# ───────────────────────────────
qat_model = models.mobilenet_v2(pretrained=False, quantize=False)
qat_model.classifier[1] = nn.Linear(qat_model.classifier[1].in_features, 200)
qat_model.load_state_dict(baseline.state_dict())          # start from FP32 weights
qat_model.fuse_model()                                    # Conv‑BN‑ReLU → fused
qat_model.qconfig = get_default_qat_qconfig("fbgemm")
prepare_qat(qat_model, inplace=True)

In [ ]:
# ───────────────────────────────
# QAT training
# ───────────────────────────────
def fine_tune_qat(model: nn.Module, train_ld, epochs=5, lr=1e-4):
    model.to(device).train()
    opt, loss_fn = optim.Adam(model.parameters(), lr), nn.CrossEntropyLoss()
    for ep in range(epochs):
        running = 0.0
        for i, (x, y) in enumerate(train_ld, 1):
            x, y = x.to(device), y.to(device)
            opt.zero_grad()
            loss = loss_fn(model(x), y)
            loss.backward(); opt.step()
            running += loss.item()
            if i % 50 == 0:
                print(f"[QAT] epoch {ep+1}/{epochs}  batch {i:3d}  loss {loss.item():.4f}")
        print(f"[QAT] epoch {ep+1} avg loss {running/len(train_ld):.4f}")

fine_tune_qat(qat_model, train_loader, epochs=5, lr=1e-4)

In [ ]:
# ───────────────────────────────
# Convert and evaluate quantized model
# ───────────────────────────────
qat_model.eval()
quant_model = convert(qat_model, inplace=False)

acc = evaluate_model(quant_model, val_loader)
print(f"\nQAT Model Validation Accuracy: {acc:.2f}%")
print("Compressed model saved successfully.")

latency, throughput, power, energy, edp = measure_inference_metrics(quant_model, val_loader)
flops, params = measure_model_size_and_flops(quant_model, input_res=(1, 3, 224, 224))

print("\nInference Metrics:")
print(f"Total inference time: {latency * 10000:.2f} s")
print(f"Total images processed: 10000")
print(f"Average latency per image: {latency * 1000:.2f} ms")
print(f"Throughput: {throughput:.2f} images/s")
print(f"Average GPU Power: {power:.2f} W")
print(f"Energy per image: {energy:.4f} J")
print(f"Energy-Delay Product (EDP): {edp:.6f} J*s")

print("\nModel Size and FLOPs:")
print(f"FLOPs: {flops:.2f} GFLOPs")
print(f"Number of parameters: {params / 1e6:.2f} million")

torch.save(quant_model.state_dict(), "/content/mobilenetv2_qat_int8.pth")
print("\nQuantized model saved to disk.")
